# 🎓 GENI YOLO Training - Ready to Run

**Your data is already augmented! 168 images ready to train.**

Just upload `geni_training_data.zip` from your Desktop and run all cells.

**FIRST: Enable GPU (Runtime → Change runtime type → T4 GPU)**

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install YOLO
!pip install -q ultralytics

In [ ]:
# Unzip training data (upload geni_training_data.zip first!)
!unzip -q geni_training_data.zip
!ls -la augmented/

In [ ]:
import json
import os
import shutil
from pathlib import Path
import random

# Load labels
with open('augmented/labels.json') as f:
    labels = json.load(f)

print(f"📊 Loaded {len(labels)} labeled images")

# Get categories
categories = sorted(list(set(l['category'] for l in labels)))
print(f"\n📋 Categories: {categories}")

# Category counts
from collections import Counter
counts = Counter(l['category'] for l in labels)
print("\n📊 By category:")
for cat, count in counts.most_common():
    print(f"   {cat}: {count}")

In [ ]:
# Create YOLO directory structure
!mkdir -p yolo_data/images/train yolo_data/images/val
!mkdir -p yolo_data/labels/train yolo_data/labels/val

# Split train/val (90/10)
random.shuffle(labels)
split_idx = int(len(labels) * 0.9)
train_labels = labels[:split_idx]
val_labels = labels[split_idx:]

print(f"✅ Split: {len(train_labels)} train, {len(val_labels)} val")

def create_yolo_label(category, categories):
    """YOLO format: class_id x_center y_center width height (normalized)"""
    class_id = categories.index(category)
    # Whole image as bounding box
    return f"{class_id} 0.5 0.5 1.0 1.0\n"

# Process train
for label in train_labels:
    src = f"augmented/{label['filename']}"
    dst = f"yolo_data/images/train/{label['filename']}"
    shutil.copy(src, dst)
    
    label_file = Path(label['filename']).stem + '.txt'
    with open(f"yolo_data/labels/train/{label_file}", 'w') as f:
        f.write(create_yolo_label(label['category'], categories))

# Process val
for label in val_labels:
    src = f"augmented/{label['filename']}"
    dst = f"yolo_data/images/val/{label['filename']}"
    shutil.copy(src, dst)
    
    label_file = Path(label['filename']).stem + '.txt'
    with open(f"yolo_data/labels/val/{label_file}", 'w') as f:
        f.write(create_yolo_label(label['category'], categories))

print("\n✅ YOLO dataset created!")

In [ ]:
# Create data.yaml
data_yaml = f"""train: yolo_data/images/train
val: yolo_data/images/val

nc: {len(categories)}
names: {categories}
"""

with open('data.yaml', 'w') as f:
    f.write(data_yaml)

print("✅ data.yaml created:")
print(data_yaml)

In [ ]:
# Train YOLO11s
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

print("🚀 Starting training...\n")

results = model.train(
    data='data.yaml',
    epochs=40,          # Good for 168 images
    imgsz=640,
    batch=16,
    device=0,
    patience=8,
    save=True,
    project='geni_model',
    name='run',
)

print("\n✅ Training complete!")

In [ ]:
# View results
from IPython.display import Image, display

print("📊 Training Results:\n")
display(Image('geni_model/run/results.png'))
display(Image('geni_model/run/confusion_matrix.png'))

In [ ]:
# Test the model
model = YOLO('geni_model/run/weights/best.pt')

results = model.predict(
    source='yolo_data/images/val',
    save=True,
    project='geni_model',
    name='test',
)

print("✅ Test predictions saved to geni_model/test/")

In [ ]:
# Download the model
!zip -r geni_pill_model.zip geni_model/run/weights/best.pt

print("✅ Model ready to download!")
print("")
print("📥 Download: geni_pill_model.zip")
print("")
print("🎯 Next: Extract best.pt and integrate into GENI backend")